# Customer Churn Prediction
## End-to-End Data Science Project

This notebook demonstrates a complete data science workflow:
1. **Data Loading & Inspection**
2. **Exploratory Data Analysis (EDA) & Visualization**
3. **Data Preprocessing & Feature Engineering**
4. **Machine Learning Models** (Logistic Regression, Decision Tree, Random Forest)
5. **Deep Learning** (Neural Network / MLP)
6. **Model Comparison & Conclusion**

**Tools Used:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-Learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

---
## 1. Data Loading & Inspection

In [ ]:
df = pd.read_csv('customer_churn.csv')
print(f'Dataset Shape: {df.shape}')
print(f'\nColumn Types:\n{df.dtypes}')
df.head(10)

In [ ]:
# Check for missing values and basic statistics
print('Missing Values:')
print(df.isnull().sum())
print('\nBasic Statistics:')
df.describe()

---
## 2. Exploratory Data Analysis (EDA) & Visualization

### 2.1 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
sns.countplot(data=df, x='Churn', ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Churn Count')
axes[0].set_xticklabels(['Not Churned (0)', 'Churned (1)'])

# Pie chart
churn_counts = df['Churn'].value_counts()
axes[1].pie(churn_counts, labels=['Not Churned', 'Churned'], autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Churn Proportion')

plt.tight_layout()
plt.show()

### 2.2 Numerical Feature Distributions

In [ ]:
numerical_cols = ['Age', 'Tenure_Months', 'Monthly_Charges', 'Total_Charges']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, col in enumerate(numerical_cols):
    ax = axes[i // 2][i % 2]
    sns.histplot(data=df, x=col, hue='Churn', kde=True, ax=ax, palette=['#2ecc71', '#e74c3c'])
    ax.set_title(f'{col} Distribution by Churn')

plt.tight_layout()
plt.show()

### 2.3 Categorical Feature Analysis

In [ ]:
categorical_cols = ['Gender', 'Contract_Type', 'Internet_Service', 'Payment_Method']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, col in enumerate(categorical_cols):
    ax = axes[i // 2][i % 2]
    ct = pd.crosstab(df[col], df['Churn'], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=ax, color=['#2ecc71', '#e74c3c'])
    ax.set_title(f'Churn Rate by {col}')
    ax.set_ylabel('Proportion')
    ax.legend(['Not Churned', 'Churned'])
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 2.4 Correlation Heatmap

In [ ]:
# Encode categorical columns for correlation analysis
df_corr = df.copy()
for col in categorical_cols:
    df_corr[col] = LabelEncoder().fit_transform(df_corr[col])

plt.figure(figsize=(10, 8))
corr_matrix = df_corr.drop('CustomerID', axis=1).corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

---
## 3. Data Preprocessing & Feature Engineering

In [ ]:
# One-hot encoding for categorical variables
df_model = pd.get_dummies(df.drop('CustomerID', axis=1), drop_first=True)

X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

# Feature Scaling (required for Logistic Regression and Neural Network)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training Set: {X_train.shape[0]} samples')
print(f'Testing Set:  {X_test.shape[0]} samples')
print(f'Churn Rate (Train): {y_train.mean():.2%}')
print(f'Churn Rate (Test):  {y_test.mean():.2%}')

---
## 4. Machine Learning Models

### 4.1 Logistic Regression

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

print('Logistic Regression Results:')
print(f'Accuracy: {accuracy_score(y_test, lr_pred):.4f}')
print(f'\n{classification_report(y_test, lr_pred)}')

### 4.2 Decision Tree Classifier

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

print('Decision Tree Results:')
print(f'Accuracy: {accuracy_score(y_test, dt_pred):.4f}')
print(f'\n{classification_report(y_test, dt_pred)}')

### 4.3 Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print('Random Forest Results:')
print(f'Accuracy: {accuracy_score(y_test, rf_pred):.4f}')
print(f'\n{classification_report(y_test, rf_pred)}')

---
## 5. Deep Learning - Neural Network (MLP)

In [ ]:
nn_model = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', solver='adam',
                         max_iter=200, random_state=42)
nn_model.fit(X_train, y_train)
nn_pred = nn_model.predict(X_test)

print('Neural Network (MLP) Results:')
print(f'Accuracy: {accuracy_score(y_test, nn_pred):.4f}')
print(f'\n{classification_report(y_test, nn_pred)}')

---
## 6. Model Comparison

### 6.1 Accuracy Comparison Bar Chart

In [ ]:
models = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'Neural Network']
accuracies = [
    accuracy_score(y_test, lr_pred),
    accuracy_score(y_test, dt_pred),
    accuracy_score(y_test, rf_pred),
    accuracy_score(y_test, nn_pred)
]

plt.figure(figsize=(10, 6))
colors = ['#3498db', '#e67e22', '#2ecc71', '#9b59b6']
bars = plt.bar(models, accuracies, color=colors, edgecolor='black', linewidth=0.5)

for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{acc:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.ylim(0, 1.05)
plt.ylabel('Accuracy Score')
plt.title('Model Accuracy Comparison')
plt.tight_layout()
plt.show()

### 6.2 ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

model_dict = {
    'Logistic Regression': lr_model,
    'Decision Tree': dt_model,
    'Random Forest': rf_model,
    'Neural Network': nn_model
}

for name, model in model_dict.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 6.3 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

predictions = [lr_pred, dt_pred, rf_pred, nn_pred]
for i, (name, pred) in enumerate(zip(models, predictions)):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(name)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.show()

---
## Conclusion

In this project, we performed a complete data science pipeline:

- **EDA** revealed that customers with month-to-month contracts, higher monthly charges, and shorter tenure are more likely to churn.
- **Logistic Regression** served as a strong baseline model with good interpretability.
- **Decision Tree** provided an easily understandable model but was prone to overfitting.
- **Random Forest** (ensemble method) improved generalization over a single decision tree.
- **Neural Network (MLP)** captured non-linear patterns in the data.

All four models were compared using Accuracy, Classification Reports, ROC Curves, and Confusion Matrices to identify the best-performing approach.